# Klasifikasi: Binary, Multi-class, dan Multi-label (9 dataset Kaggle)

| Jenis | Aturan | Jumlah dataset |
|---|---|---|
| **Binary** | tepat **2 kelas**, 1 label per sampel | 3 |
| **Multi-class** | **lebih dari 2 kelas**, tetap 1 label per sampel | 3 |
| **Multi-label** | **lebih dari 2 label**, 1 sampel boleh punya banyak label | 3 |


In [1]:
import ast, glob
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler

RINGKASAN = []  # diisi tiap dataset, ditampilkan di akhir

def find(filename):
    # cari file berdasarkan nama (tanpa peduli huruf besar/kecil) di folder ini, lalu folder induk
    import os
    for root in [".", ".."]:
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d != "node_modules"]
            for f in files:
                if f.lower() == filename.lower():
                    return os.path.join(dirpath, f)
    raise FileNotFoundError(f"{filename} tidak ditemukan. Taruh notebook di folder induk yang berisi folder-folder datasetnya.")

def read_csv_auto(path, **kw):
    # baca CSV; kalau hasilnya 1 kolom, coba lagi dengan pemisah titik koma
    df = pd.read_csv(path, **kw)
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=";", **kw)
    return df

def check_binary(name, y, url):
    n = y.nunique(); ok = n == 2
    print(f"[binary] {name}: {n} kelas {sorted(y.unique().tolist())} -> sesuai (tepat 2)? {ok}")
    print(f"         Distribusi: {y.value_counts().to_dict()}")
    print(f"         Sumber    : {url}\n")
    RINGKASAN.append(("binary", name, n, ok, url))

def check_multiclass(name, y, url):
    n = y.nunique(); ok = n > 2
    print(f"[multi-class] {name}: {n} kelas {sorted(y.unique().tolist())} -> sesuai (>2)? {ok}")
    print(f"              Distribusi: {y.value_counts().sort_index().to_dict()}")
    print(f"              Sumber    : {url}\n")
    RINGKASAN.append(("multi-class", name, n, ok, url))

def check_multilabel(name, Y, url):
    Y = pd.DataFrame(Y)
    n_labels = Y.shape[1]; per_row = Y.sum(axis=1)
    ok = bool(n_labels > 2 and per_row.max() > 1)
    print(f"[multi-label] {name}: {n_labels} label -> lebih dari 2? {n_labels > 2}")
    print(f"              Label per sampel: min={per_row.min()}, maks={per_row.max()}, rata-rata={per_row.mean():.2f}")
    print(f"              Ada sampel dengan >1 label (benar multi-label)? {per_row.max() > 1}")
    print(f"              Sumber: {url}\n")
    RINGKASAN.append(("multi-label", name, n_labels, ok, url))

def baseline_tabular(X, y, cv=5):
    num = X.select_dtypes("number").columns.tolist()
    cat = [c for c in X.columns if c not in num]
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat),
    ])
    model = make_pipeline(pre, LogisticRegression(max_iter=1000))
    print(f"Akurasi CV {cv}-fold:", cross_val_score(model, X, y, cv=cv).mean().round(3))

def baseline_text(texts, Y, n=20000):
    texts, Y = texts.iloc[:n], Y.iloc[:n]
    Xtr, Xte, Ytr, Yte = train_test_split(texts, Y, test_size=0.2, random_state=42)
    vec = TfidfVectorizer(max_features=20000)
    clf = OneVsRestClassifier(LogisticRegression(max_iter=500)).fit(vec.fit_transform(Xtr), Ytr)
    pred = clf.predict(vec.transform(Xte))
    print("F1 macro:", round(f1_score(Yte, pred, average="macro", zero_division=0), 3))

def baseline_tabular_multilabel(X, Y):
    num = X.select_dtypes("number").columns.tolist()
    cat = [c for c in X.columns if c not in num]
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat),
    ])
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=42)
    clf = make_pipeline(pre, OneVsRestClassifier(LogisticRegression(max_iter=1000))).fit(Xtr, Ytr)
    print("F1 macro:", round(f1_score(Yte, clf.predict(Xte), average="macro", zero_division=0), 3))

def label_columns(df, exclude=()):
    return [c for c in df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df[c]) and set(df[c].dropna().unique()) <= {0, 1}]

# BAGIAN 1: Binary

## 1.1 SMS Spam Collection
- Sumber: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset (diunggah UCI Machine Learning di Kaggle; 5.574 pesan SMS berbahasa Inggris)
- Target: `v1` (ham = pesan biasa, spam = pesan sampah); teks pesan ada di kolom `v2`

In [2]:
URL = "https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset"
df = pd.read_csv(find("spam.csv"), encoding="latin-1")[["v1", "v2"]]
y, X = df["v1"], df["v2"]
check_binary("SMS Spam Collection", y, URL)
model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
print("Akurasi CV 5-fold:", cross_val_score(model, X, y, cv=5).mean().round(3))

[binary] SMS Spam Collection: 2 kelas ['ham', 'spam'] -> sesuai (tepat 2)? True
         Distribusi: {'ham': 4825, 'spam': 747}
         Sumber    : https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

Akurasi CV 5-fold: 0.97


## 1.2 Bank Marketing
- Sumber: https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset (diunggah janiobachmann; kampanye telemarketing bank, 11.162 baris, 17 kolom)
- Target: `deposit` (yes / no: apakah nasabah membuka deposito berjangka)

In [3]:
URL = "https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset"
df = read_csv_auto(find("bank.csv"))
target = "deposit" if "deposit" in df.columns else "y"
y = df[target]
X = df.drop(columns=[target])
check_binary("Bank Marketing", y, URL)
baseline_tabular(X, y)

[binary] Bank Marketing: 2 kelas ['no', 'yes'] -> sesuai (tepat 2)? True
         Distribusi: {'no': 5873, 'yes': 5289}
         Sumber    : https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset

Akurasi CV 5-fold: 0.784


## 1.3 Telco Customer Churn
- Sumber: https://www.kaggle.com/datasets/blastchar/telco-customer-churn (diunggah blastchar; 7.043 pelanggan, 21 kolom)
- Target: `Churn` (Yes / No)

In [4]:
URL = "https://www.kaggle.com/datasets/blastchar/telco-customer-churn"
df = read_csv_auto(find("WA_Fn-UseC_-Telco-Customer-Churn.csv"))
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # ada nilai kosong berupa spasi
y = df["Churn"]
X = df.drop(columns=["Churn", "customerID"])
check_binary("Telco Customer Churn", y, URL)
baseline_tabular(X, y)

[binary] Telco Customer Churn: 2 kelas ['No', 'Yes'] -> sesuai (tepat 2)? True
         Distribusi: {'No': 5174, 'Yes': 1869}
         Sumber    : https://www.kaggle.com/datasets/blastchar/telco-customer-churn

Akurasi CV 5-fold: 0.804


# BAGIAN 2: Multi-class


## 2.1 Palmer Penguins
- Sumber: https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data (data penguin dari Dr. Kristen Gorman dan Palmer Station LTER, Antartika)
- Target: `species` (Adelie, Chinstrap, Gentoo)

In [5]:
URL = "https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data"
df = pd.read_csv(find("penguins_size.csv")).dropna(subset=["species"])
y = df["species"]
X = df.drop(columns=["species"])
check_multiclass("Palmer Penguins", y, URL)
baseline_tabular(X, y)

[multi-class] Palmer Penguins: 3 kelas ['Adelie', 'Chinstrap', 'Gentoo'] -> sesuai (>2)? True
              Distribusi: {'Adelie': 152, 'Chinstrap': 68, 'Gentoo': 124}
              Sumber    : https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data

Akurasi CV 5-fold: 0.994


## 2.2 Mobile Price Classification
- Sumber: https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification (diunggah iabhishekofficial; spesifikasi ponsel)
- Target: `price_range` (0 = murah, 1 = menengah, 2 = mahal, 3 = sangat mahal)

In [6]:
URL = "https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification"
df = pd.read_csv(find("train.csv"))
assert "price_range" in df.columns, "train.csv ini bukan Mobile Price (kolom price_range tidak ada)"
y = df["price_range"]
X = df.drop(columns=["price_range"])
check_multiclass("Mobile Price Classification", y, URL)
baseline_tabular(X, y)

[multi-class] Mobile Price Classification: 4 kelas [0, 1, 2, 3] -> sesuai (>2)? True
              Distribusi: {0: 500, 1: 500, 2: 500, 3: 500}
              Sumber    : https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification

Akurasi CV 5-fold: 0.964


## 2.3 Fashion-MNIST
- Sumber: https://www.kaggle.com/datasets/zalando-research/fashionmnist (Zalando Research; gambar 28x28 piksel dalam format CSV)
- Target: `label` (10 kelas: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot)

In [7]:
URL = "https://www.kaggle.com/datasets/zalando-research/fashionmnist"
df = pd.read_csv(find("fashion-mnist_train.csv")).sample(5000, random_state=42)  # sampel agar cepat
y = df["label"]
X = df.drop(columns=["label"]) / 255.0
check_multiclass("Fashion-MNIST", y, URL)
baseline_tabular(X, y, cv=3)

[multi-class] Fashion-MNIST: 10 kelas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] -> sesuai (>2)? True
              Distribusi: {0: 486, 1: 481, 2: 547, 3: 510, 4: 494, 5: 480, 6: 518, 7: 499, 8: 459, 9: 526}
              Sumber    : https://www.kaggle.com/datasets/zalando-research/fashionmnist

Akurasi CV 3-fold: 0.798


# BAGIAN 3: Multi-label

## 3.1 Netflix Movies and TV Shows (genre)
- Sumber: https://www.kaggle.com/datasets/shivamb/netflix-shows (diunggah shivamb; sekitar 8.800 judul, file `netflix_titles.csv` sekitar 3,4 MB)
- Label: kolom `listed_in` (daftar genre per judul, dipisah koma), diubah jadi matriks 0/1; dipakai 20 genre terbanyak
- Fitur: teks `description`

In [8]:
URL = "https://www.kaggle.com/datasets/shivamb/netflix-shows"
df = pd.read_csv(find("netflix_titles.csv")).dropna(subset=["listed_in", "description"])
genre_list = df["listed_in"].str.split(",").apply(lambda g: [x.strip() for x in g])
top20 = pd.Series([x for g in genre_list for x in g]).value_counts().head(20).index.tolist()
mlb = MultiLabelBinarizer(classes=top20)
Y = pd.DataFrame(mlb.fit_transform(genre_list), columns=top20, index=df.index)
check_multilabel("Netflix (genre)", Y, URL)
baseline_text(df["description"], Y)

[multi-label] Netflix (genre): 20 label -> lebih dari 2? True
              Label per sampel: min=0, maks=3, rata-rata=1.92
              Ada sampel dengan >1 label (benar multi-label)? True
              Sumber: https://www.kaggle.com/datasets/shivamb/netflix-shows



c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['Anime Features', 'Anime Series', 'British TV Shows', 'Classic & Cult TV', 'Classic Movies', 'Cult Movies', 'Faith & Spirituality', 'Korean TV Shows', 'LGBTQ Movies', 'Movies', 'Sci-Fi & Fantasy', 'Science & Nature TV', 'Spanish-Language TV Shows', 'Sports Movies', 'Stand-Up Comedy & Talk Shows', 'TV Action & Adventure', 'TV Horror', 'TV Mysteries', 'TV Sci-Fi & Fantasy', 'TV Shows', 'TV Thrillers', 'Teen TV Shows'] will be ignored
  warnings.warn(


F1 macro: 0.09


## 3.2 Pokemon (tipe)
- Sumber: https://www.kaggle.com/datasets/abcsds/pokemon (diunggah abcsds; 800 Pokemon, lisensi CC0)
- Label: `Type 1` dan `Type 2` digabung jadi satu set tipe (satu Pokemon bisa punya 1 atau 2 tipe)
- Fitur: statistik (HP, Attack, Defense, Sp. Atk, Sp. Def, Speed, Generation, Legendary)

In [9]:
URL = "https://www.kaggle.com/datasets/abcsds/pokemon"
df = pd.read_csv(find("Pokemon.csv"))
type_list = df[["Type 1", "Type 2"]].apply(lambda r: [t for t in r if pd.notna(t)], axis=1)
mlb = MultiLabelBinarizer()
Y = pd.DataFrame(mlb.fit_transform(type_list), columns=mlb.classes_, index=df.index)
X = df[["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Generation", "Legendary"]].astype(float)
check_multilabel("Pokemon (tipe)", Y, URL)
baseline_tabular_multilabel(X, Y)

[multi-label] Pokemon (tipe): 18 label -> lebih dari 2? True
              Label per sampel: min=1, maks=2, rata-rata=1.52
              Ada sampel dengan >1 label (benar multi-label)? True
              Sumber: https://www.kaggle.com/datasets/abcsds/pokemon

F1 macro: 0.038


## 3.3 Anime Recommendations Database (genre)
- Sumber: https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database (data dari myanimelist.net; hanya file `anime.csv` yang dipakai, sekitar 12.000 anime)
- Label: kolom `genre` (daftar genre dipisah koma), dipakai 20 genre terbanyak
- Fitur: `type`, `episodes`, `rating`, `members`

In [10]:
URL = "https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database"
df = pd.read_csv(find("anime.csv")).dropna(subset=["genre"])
df["episodes"] = pd.to_numeric(df["episodes"], errors="coerce")  # ada nilai "Unknown"
genre_list = df["genre"].str.split(",").apply(lambda g: [x.strip() for x in g])
top20 = pd.Series([x for g in genre_list for x in g]).value_counts().head(20).index.tolist()
mlb = MultiLabelBinarizer(classes=top20)
Y = pd.DataFrame(mlb.fit_transform(genre_list), columns=top20, index=df.index)
X = df[["type", "episodes", "rating", "members"]]
check_multilabel("Anime (genre)", Y, URL)
baseline_tabular_multilabel(X, Y)

[multi-label] Anime (genre): 20 label -> lebih dari 2? True
              Label per sampel: min=0, maks=10, rata-rata=2.52
              Ada sampel dengan >1 label (benar multi-label)? True
              Sumber: https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database



c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['Cars', 'Dementia', 'Demons', 'Game', 'Harem', 'Horror', 'Josei', 'Martial Arts', 'Military', 'Mystery', 'Parody', 'Police', 'Psychological', 'Samurai', 'Shoujo Ai', 'Shounen Ai', 'Space', 'Sports', 'Super Power', 'Thriller', 'Vampire', 'Yaoi', 'Yuri'] will be ignored
  warnings.warn(


F1 macro: 0.067


# Ringkasan

In [11]:
pd.DataFrame(RINGKASAN, columns=["jenis", "dataset", "jumlah_kelas_atau_label", "sesuai_aturan", "sumber"])

,jenis,dataset,jumlah_kelas_atau_label,sesuai_aturan,sumber
0,binary,SMS Spam Collection,2,True,https://www.kaggle.com/datasets/uciml/sms-spam...
1,binary,Bank Marketing,2,True,https://www.kaggle.com/datasets/janiobachmann/...
2,binary,Telco Customer Churn,2,True,https://www.kaggle.com/datasets/blastchar/telc...
3,multi-class,Palmer Penguins,3,True,https://www.kaggle.com/datasets/parulpandey/pa...
4,multi-class,Mobile Price Classification,4,True,https://www.kaggle.com/datasets/iabhishekoffic...
5,multi-class,Fashion-MNIST,10,True,https://www.kaggle.com/datasets/zalando-resear...
6,multi-label,Netflix (genre),20,True,https://www.kaggle.com/datasets/shivamb/netfli...
7,multi-label,Pokemon (tipe),18,True,https://www.kaggle.com/datasets/abcsds/pokemon
8,multi-label,Anime (genre),20,True,https://www.kaggle.com/datasets/CooperUnion/an...
